In [2]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import glob
import os
import re
import gzip

In [ ]:
me_bed = glob.glob("/working/lab_olgak/lijunX/OV_meBRCA1_Nanopore/output/*/Me/*_m_pileup_tab.bed")

df_list = []
for file_path in me_bed:
    sample_name = re.search(r'output/(.*)/Me/', file_path).group(1)
    print(sample_name)

    chunk_list = []
    for chunk in pd.read_csv(file_path, sep='\t', header=None, chunksize = 100000):
        chunk.columns = ["chrom", "start", "end", "mod_type", "Nvalid_cov_2", "strand", "start_2", "end_2", "color_2", "Nvalid_cov", "fraction_mod", "Nmod", "Ncanonical", "Nother_mod", "Ndelete", "Nfail", "Ndiff", "Nnocall"]
        valid_chromosomes = [f'chr{i}' for i in range(1, 23)]
        chunk = chunk[chunk['chrom'].isin(valid_chromosomes)]
        if len(chunk) > 500:
            chunk = chunk.sample(n=1000)
        chunk_list.append(chunk)
    file_df = pd.concat(chunk_list, ignore_index=True)
    file_df['sample'] = sample_name
    df_list.append(file_df)


methylation_5mC = pd.concat(df_list, ignore_index=True)

sample_to_group = {"OVCAR8":"het_meBRCA1","134048_7794_T2":"het_meBRCA1", "134169_6606_T2":"het_meBRCA1",
                   "SFRC01428S1_22235_T2":"het_meBRCA1", "SFRC01495S2_1537_T1":"het_meBRCA1", "SOLACE2_0126S1_25285_T2":"het_meBRCA1",
                   "WEHI_CS62":"hom_meBRCA1", "OVCAR8 LP_A6_30":"hom_meBRCA1", "134011_1910_T4":"hom_meBRCA1",
                   "OVCAR8_LP_A6_30": "hom_meBRCA1", "134062_9174_T4":"hom_meBRCA1", "SFRC01585S1_2163_T3":"hom_meBRCA1",
                   "UWB1": "mut_BRCA1", "UWB1_BRCA1": "func_BRCA1"}
methylation_5mC['group'] = methylation_5mC['sample'].map(sample_to_group)

methylation_5mC.to_csv("/working/lab_olgak/lijunX/OV_meBRCA1_Nanopore/intermediate/sampled_5mC.csv", encoding='utf-8', index=False)


In [ ]:
me_bed = glob.glob("/working/lab_olgak/lijunX/OV_meBRCA1_Nanopore/output/*/Me/*_pileup_h.bed.gz")

with gzip.open(file_path, "rt") as f:

    df_list = []
    for file_path in me_bed:
        sample_name = re.search(r'output/(.*)/Me/', file_path).group(1)
        print(sample_name)
        chunk_list = []
        for chunk in pd.read_csv(file_path, sep='\t', header=None, chunksize = 100000):
            chunk.columns = ["chrom", "start", "end", "mod_type", "Nvalid_cov_2", "strand", "start_2", "end_2", "color_2", "Nvalid_cov", "fraction_mod", "Nmod", "Ncanonical", "Nother_mod", "Ndelete", "Nfail", "Ndiff", "Nnocall"]
            valid_chromosomes = [f'chr{i}' for i in range(1, 23)]
            chunk = chunk[chunk['chrom'].isin(valid_chromosomes)]
            if len(chunk) > 500:
                chunk = chunk.sample(n=1000)
            chunk_list.append(chunk)
        file_df = pd.concat(chunk_list, ignore_index=True)
        file_df['sample'] = sample_name
        df_list.append(file_df)


methylation_5hmC = pd.concat(df_list, ignore_index=True)

sample_to_group = {"OVCAR8":"het_meBRCA1","134048_7794_T2":"het_meBRCA1", "134169_6606_T2":"het_meBRCA1",
                   "SFRC01428S1_22235_T2":"het_meBRCA1", "SFRC01495S2_1537_T1":"het_meBRCA1", "SOLACE2_0126S1_25285_T2":"het_meBRCA1",
                   "WEHI_CS62":"hom_meBRCA1", "OVCAR8 LP_A6_30":"hom_meBRCA1", "134011_1910_T4":"hom_meBRCA1",
                   "OVCAR8_LP_A6_30": "hom_meBRCA1", "134062_9174_T4":"hom_meBRCA1", "SFRC01585S1_2163_T3":"hom_meBRCA1",
                   "UWB1": "mut_BRCA1", "UWB1_BRCA1": "func_BRCA1"}
methylation_5hmC['group'] = methylation_5hmC['sample'].map(sample_to_group)

print(methylation_5hmC.head())
methylation_5hmC.to_csv("/working/lab_olgak/lijunX/OV_meBRCA1_Nanopore/output/sampled_5hmC.csv", encoding='utf-8', index=False)


In [6]:
island_pd = pd.read_csv("/working/lab_olgak/lijunX/OV_meBRCA1_Nanopore/ref/cpg_islands_ucsc_cleaned.bed", sep = "\t")
island_pd.columns = ["chrom", "i_start", "i_end", "name"]

valid_chromosomes = [f'chr{i}' for i in range(1, 23)]
island_pd = island_pd[island_pd['chrom'].isin(valid_chromosomes)]

island_extended = island_pd.copy()
island_extended["i_start"] = island_extended["i_start"] - 4000
island_extended["i_end"] = island_extended["i_end"] + 4000

print(island_extended)

      chrom    i_start      i_end      name
0      chr1     131124     139563   CpG: 30
1      chr1     195251     204121  CpG: 104
2      chr1     364792     374063   CpG: 99
3      chr1     377172     386185   CpG: 84
4      chr1     487107     495546   CpG: 29
...     ...        ...        ...       ...
30682  chr9  138042856  138051074   CpG: 18
30683  chr9  138143698  138152159   CpG: 49
30684  chr9  138146056  138154661   CpG: 61
30685  chr9  138170943  138179739   CpG: 62
30686  chr9  138216919  138225140   CpG: 27

[26865 rows x 4 columns]


In [5]:
me_bed = glob.glob("/working/lab_olgak/lijunX/OV_meBRCA1_Nanopore/output/*/Me/*_m_pileup_tab.bed")

df_list = []
for file_path in me_bed:
    sample_name = re.search(r'output/(.*)/Me/', file_path).group(1)
    print(sample_name)

    chunk_list = []
    for chunk in pd.read_csv(file_path, sep='\t', header=None, chunksize = 100000):
        chunk.columns = ["chrom", "start", "end", "mod_type", "Nvalid_cov_2", "strand", "start_2", "end_2", "color_2", "Nvalid_cov", "fraction_mod", "Nmod", "Ncanonical", "Nother_mod", "Ndelete", "Nfail", "Ndiff", "Nnocall"]
        valid_chromosomes = [f'chr{i}' for i in range(1, 23)]
        chunk = chunk[chunk['chrom'].isin(valid_chromosomes)]
        chunk.merge(island_extended, how ="inner", on = "chrom")
        .query( "start >=i_start & end <= i_end")
        print(chunk.head())
        break
        chunk_list.append(chunk)
    break
    file_df = pd.concat(chunk_list, ignore_index=True)
    file_df['sample'] = sample_name
    df_list.append(file_df)


methylation_5mC = pd.concat(df_list, ignore_index=True)

SOLACE2_0126S1_25285_T2
  chrom  start    end mod_type  Nvalid_cov_2 strand  start_2  end_2  color_2  \
0  chr1  10468  10469        m            39      .    10468  10469  255,0,0   
1  chr1  10470  10471        m            35      .    10470  10471  255,0,0   
2  chr1  10483  10484        m            20      .    10483  10484  255,0,0   
3  chr1  10488  10489        m            48      .    10488  10489  255,0,0   
4  chr1  10492  10493        m            51      .    10492  10493  255,0,0   

   Nvalid_cov  fraction_mod  Nmod  Ncanonical  Nother_mod  Ndelete  Nfail  \
0          39         56.41    22          17           0        0     18   
1          35         57.14    20          15           0        1     16   
2          20         80.00    16           4           0        9     28   
3          48         45.83    22          26           0        0     10   
4          51         43.14    22          29           0        1      5   

   Ndiff  Nnocall  
0      0    

ValueError: No objects to concatenate